In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ml-challenge/embeddings_2.npy
/kaggle/input/ml-challenge/embeddings_1.npy
/kaggle/input/ml-challenge/test_data.npy
/kaggle/input/ml-challenge/icd_codes_1.txt
/kaggle/input/ml-challenge/icd_codes_2.txt
/kaggle/input/ml-challenge/sample_solution.csv


In [11]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd
from tensorflow.keras.layers import LeakyReLU

# File paths
embedding_file_1 = '/kaggle/input/ml-challenge/embeddings_1.npy'
label_file_1 = '/kaggle/input/ml-challenge/icd_codes_1.txt'
embedding_file_2 = '/kaggle/input/ml-challenge/embeddings_2.npy'
label_file_2 = '/kaggle/input/ml-challenge/icd_codes_2.txt'
test_embedding_file = '/kaggle/input/ml-challenge/test_data.npy'
output_file = 'predictions.csv'

# Embeddings
embeddings_1 = np.load(embedding_file_1)
embeddings_2 = np.load(embedding_file_2)
embeddings = np.vstack([embeddings_1, embeddings_2])  # Combine chunks

# Labels
def load_labels(file_path):
    with open(file_path, 'r') as f:
        labels = [line.strip().replace("'", "").split(';') for line in f.readlines()]
    return labels

labels_1 = load_labels(label_file_1)
labels_2 = load_labels(label_file_2)
labels = labels_1 + labels_2

# Encoding labels
mlb = MultiLabelBinarizer()
multi_hot_labels = mlb.fit_transform(labels)

# Data Split
X_train, X_val, y_train, y_val = train_test_split(embeddings, multi_hot_labels, test_size=0.0085, random_state=42)



In [12]:
# Scoring metric as per problem statement
class F2Score(tf.keras.metrics.Metric):
    def __init__(self, name="f2_score", threshold=0.5, **kwargs):
        super(F2Score, self).__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name="tp", initializer="zeros")
        self.false_positives = self.add_weight(name="fp", initializer="zeros")
        self.false_negatives = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)  
        y_true = tf.cast(y_true, tf.float32) 

        tp = tf.reduce_sum(y_true * y_pred)  # True positives
        fp = tf.reduce_sum((1 - y_true) * y_pred)  # False positives
        fn = tf.reduce_sum(y_true * (1 - y_pred))  # False negatives

        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        f2_score = (5 * precision * recall) / (4 * precision + recall + tf.keras.backend.epsilon())
        return f2_score
    
    def reset_states(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)
        

In [13]:
# Model
base_model_1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    
    tf.keras.layers.Dense(1024),
    LeakyReLU(negative_slope=0.175),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5), 
    
    tf.keras.layers.Dense(700),
    LeakyReLU(negative_slope=0.175),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5), 
    
    tf.keras.layers.Dense(1400, activation='sigmoid') 
])

# Training
base_model_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00009), 
                     loss='binary_crossentropy', 
                     metrics=[F2Score()])
base_model_1.fit(X_train, y_train, epochs=150, batch_size=128, validation_data=(X_val, y_val))

# Testing
test_embeddings = np.load(test_embedding_file)
final_predictions = base_model_1.predict(test_embeddings)

# Final predictions
final_predictions_binary = (final_predictions > 0.5).astype(int)
predicted_labels = mlb.inverse_transform(final_predictions_binary)

# Saving results
predicted_codes_str = [";".join(codes) for codes in predicted_labels]
submission_df = pd.DataFrame({
    'id': range(1, len(predicted_codes_str) + 1),
    'labels': predicted_codes_str
})
submission_df.to_csv(output_file, index=False)

print("Predictions saved to", output_file)

Epoch 1/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - f2_score: 0.0120 - loss: 0.6025 - val_f2_score: 0.1592 - val_loss: 0.0204
Epoch 2/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.1731 - loss: 0.0243 - val_f2_score: 0.2330 - val_loss: 0.0069
Epoch 3/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.2563 - loss: 0.0087 - val_f2_score: 0.3241 - val_loss: 0.0050
Epoch 4/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.3460 - loss: 0.0059 - val_f2_score: 0.4206 - val_loss: 0.0039
Epoch 5/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.4408 - loss: 0.0045 - val_f2_score: 0.5167 - val_loss: 0.0030
Epoch 6/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.5249 - loss: 0.0036 - val_f2_score: 0.5908 - val_loss: 0.0025
Epoch 7/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f2_score: 0.5856 - loss: 0.0030 - val_f2_score: 0.6467 - val_loss: 0.0022
Epoch 8/150
1539/1539 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - f2_score: 0.6260 - loss: 